# Randomised Complete Block (RCB) Designs

## Overview
Blocking removes a known source of nuisance variation (spatial gradient, time of processing, observer, batch), increasing power to detect treatment effects. In a **randomised complete block design**, every treatment appears exactly once in each block.

**When to block:**
- Units grouped by a factor expected to affect the response (spatial heterogeneity, temporal batches, individual animals)
- The blocking factor is not itself of interest — it is a nuisance variable
- Quinn & Keough (2002) ch. 10 / Underwood (1997) ch. 12

**Related designs covered:**
- RCB (this notebook)
- Latin squares (treatments in rows AND columns of a 2D grid)
- Tukey's test for non-additivity (tests whether block × treatment interaction exists)

---

In [ ]:
library(tidyverse); library(emmeans); library(lme4); library(lmerTest)
set.seed(44)
# RCB experiment: 4 grazing treatments, 8 blocks (spatial positions along gradient)
# Each block contains one plot of each treatment
n_blocks <- 8; n_treat <- 4
treatments <- c("Control","Low","Medium","High")
dat <- expand.grid(
  block     = factor(1:n_blocks),
  treatment = factor(treatments, levels=treatments)
)
# Block effect (e.g., elevation gradient: higher blocks have lower productivity)
block_eff <- rep(seq(5, -5, length.out=n_blocks), times=n_treat)
treat_eff <- rep(c(0,-3,-6,-10), each=n_blocks)
dat$biomass <- 30 + treat_eff + block_eff + rnorm(nrow(dat), 0, 2)
cat("RCB dataset: 4 treatments × 8 blocks\n")
print(head(dat, 8))

---
## Classic RCB analysis and Tukey non-additivity test

In [ ]:
# RCB ANOVA — block as additive factor (no block × treatment interaction)
m_rcb <- aov(biomass ~ treatment + block, data=dat)
cat("RCB ANOVA (block as fixed factor, additive model):\n")
print(summary(m_rcb))

# How much variance did blocking explain?
ss_total  <- sum((dat$biomass - mean(dat$biomass))^2)
ss_blocks <- summary(m_rcb)[[1]]["block","Sum Sq"]
cat("\nVariance explained by blocks:", round(100*ss_blocks/ss_total, 1), "%\n")
cat("This is variance that would have inflated the error in a CRD.\n")

# Tukey's 1-df test for non-additivity
# If significant: block × treatment interaction exists → RCB assumptions violated
library(agricolae)
cat("\nTukey's test for non-additivity (block × treatment interaction):\n")
print(nonadditivity(dat$biomass, dat$treatment, dat$block, MSE=deviance(m_rcb)/df.residual(m_rcb)))

In [ ]:
# Post-hoc comparisons of treatments
em <- emmeans(m_rcb, ~ treatment)
cat("Estimated marginal means (adjusted for blocks):\n"); print(em)
cat("\nPairwise Tukey comparisons:\n")
print(pairs(em, adjust="tukey"))

# Compare: CRD (ignoring blocks) vs RCB
m_crd <- aov(biomass ~ treatment, data=dat)
cat("\nError MS comparison:\n")
cat("  CRD  error MS:", round(summary(m_crd)[[1]]["Residuals","Mean Sq"],3), "\n")
cat("  RCB  error MS:", round(summary(m_rcb)[[1]]["Residuals","Mean Sq"],3), "\n")
cat("  Blocking efficiency: blocks removed",
    round(100*(1-summary(m_rcb)[[1]]["Residuals","Mean Sq"]/
              summary(m_crd)[[1]]["Residuals","Mean Sq"]),1),
    "% of error variance\n")

In [ ]:
# Latin square: blocks in two dimensions (row AND column)
# Each treatment appears once per row and once per column
cat("=== Latin Square Design ===\n")
# 4×4 Latin square
latin_sq <- data.frame(
  row   = rep(1:4, each=4),
  col   = rep(1:4, times=4),
  treat = factor(c("A","B","C","D", "B","C","D","A",
                   "C","D","A","B", "D","A","B","C"))
)
row_eff <- rep(c(2,0,-1,-3), each=4)
col_eff <- rep(c(1,-1,0,2), times=4)
treat_eff_ls <- c(A=0, B=-4, C=-7, D=-11)[as.character(latin_sq$treat)]
latin_sq$y <- 20 + treat_eff_ls + row_eff + col_eff + rnorm(16,0,1.5)

m_ls <- aov(y ~ treat + factor(row) + factor(col), data=latin_sq)
cat("Latin square ANOVA (controls row AND column gradients):\n")
print(summary(m_ls))
cat("\nLatin square: n² observations test n treatments,\n")
cat("removing two orthogonal blocking factors simultaneously.\n")

---
## Common Pitfalls

**1. Omitting the block term from the ANOVA**
If blocks are not included in the model, their effect inflates the error term, reducing power. The error MS will be larger than necessary, and the F-ratio for treatments will be smaller than it should be. Always include the blocking factor.

**2. Using block as a random effect when blocks are not a random sample**
If blocks are selected for systematic reasons (specific positions along a gradient, known batches), treating them as fixed effects is more appropriate. Treat blocks as random only when they are a random sample from a larger population of possible blocks.

**3. Ignoring a significant Tukey non-additivity test**
A significant non-additivity test means the block × treatment interaction is non-negligible — the treatment effect differs across blocks. The simple additive RCB model is then invalid. Options: transform the response, use a mixed model with the interaction, or acknowledge the issue in interpretation.

**4. Blocking on a variable that explains little variance**
Blocking reduces error degrees of freedom (losing one df per block minus one). If the blocking variable is unrelated to the response, this loss of df reduces rather than increases power. Block only when the blocking factor is expected to explain substantial variation (> ~10% of total variance).

**5. Not randomising treatments within blocks**
Randomised complete block design requires random assignment of treatments within each block. Systematic assignment (always the same treatment in the same position within a block) introduces confounding and invalidates the error term.

**6. Applying RCB when blocks are too small**
A complete block must contain one replicate of every treatment. If blocks are spatially small relative to treatment-scale variation, all treatments within a block may be similarly affected, defeating the purpose of blocking. Match block size to the scale of the nuisance variation.


---
*r_methods_library - Samantha McGarrigle*